In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from web_scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [18]:
load_dotenv(override=True)
api_key=os.getenv("GEMINI_API_KEY")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
model = "gemini-2.5-flash"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key = api_key)

In [3]:
links = fetch_website_links("https://edwarddonner.com")
print(links)

['https://edwarddonner.com/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/', 

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://adk.dev"))


Here is the list of links on the website https://adk.dev -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/2.0/
https://github.com/google/adk-js/releases/tag/adk-v1.0.0
.
.
https://github.com/google/adk-python
https://github.com/google/adk-js
https://github.com/google/adk-go
https://github.com/google/adk-java
.
get-started/
runtime/
get-started/about/
integrations/
api-reference/
community/
2.0/
https://github.com/google/adk-python
https://github.com/google/adk-js
https://github.com/google/adk-go
https://github.com/google/adk-java
.
get-started/
get-started/python/
get-started/typescript/
get-started/go/
get-started/java/
tutorials/
tutorials/multi-tool-agent/
tutorials/agent-team/
get-started/streaming/
get-started/streaming/quickstart-streaming/
get-started/streaming/quickstart-streaming-java/
visual-bu

In [17]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [19]:
select_relevant_links("https://krnetworkcloud.org/?srsltid=AfmBOorA389GvBlqIEVvPe1CXyYztOaRPnX54Nmr4YP5jZIq6eupiu94")

{'links': [{'type': 'about page', 'url': 'https://krnetworkcloud.org/about/'},
  {'type': 'contact page', 'url': 'https://krnetworkcloud.org/contact/'},
  {'type': 'all courses overview',
   'url': 'https://krnetworkcloud.org/courses/'},
  {'type': 'course type: redhat technologies',
   'url': 'https://krnetworkcloud.org/course-type/redhat-technologies/'},
  {'type': 'course type: devops and cloud technologies',
   'url': 'https://krnetworkcloud.org/course-type/devops-cloud-technologies/'},
  {'type': 'course type: cisco and microsoft technologies',
   'url': 'https://krnetworkcloud.org/course-type/cisco-and-microsoft-technologies/'},
  {'type': 'course type: python and data science',
   'url': 'https://krnetworkcloud.org/course-type/python-and-data-science/'},
  {'type': 'course type: cyber security technologies',
   'url': 'https://krnetworkcloud.org/course-type/cyber-security-technologies/'},
  {'type': 'course type: software technologies',
   'url': 'https://krnetworkcloud.org/cour

Building the brochure now!

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents=fetch_website_contents(url)
    relevant_links=select_relevant_links(url)
    results=f"##Landing Page:\n\n{contents}\nRelevant Links:\n"
    for link in relevant_links['links']:
        results+=f"\n\n ##Link: {link['type']}\n"
        results+=fetch_website_contents(link['url'])
    return results
    

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

##Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
123k
•
2.74k
moonshotai/Kimi-K2.6
Updated
3 days ago
•
376k
•
1.04k
Qwen/Qwen3.6-27B
Updated
2 days ago
•
330k
•
832
openai/privacy-filter
Updated
4 days ago
•
35.8k
•
793
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
46k
•
703
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
172
ML Intern
🤖
172
Chat with an AI‑powered ML Intern assistant
Running
on
Zero
MCP
2.37k
Wan2.2 14B Preview
🐌
2.37k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
692
OmniVoice
🌍
692
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
Featured
1.01k
Fi

In [24]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n##Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n2 days ago\n•\n123k\n•\n2.74k\nmoonshotai/Kimi-K2.6\nUpdated\n3 days ago\n•\n376k\n•\n1.04k\nQwen/Qwen3.6-27B\nUpdated\n2 days ago\n•\n330k\n•\n832\nopenai/privacy-filter\nUpdated\n4 days ago\n•\n35.8k\n•\n793\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n2 days ago\n•\n46k\n•\n703\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n172\nML Intern\n🤖\n17

In [31]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-pro",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face

### The AI Community Building the Future.

Welcome to Hugging Face, the platform where the machine learning community collaborates on models, datasets, and applications. We are the home of machine learning, empowering creators and organizations to build, share, and deploy state-of-the-art AI.

---

### The Home of Machine Learning

Hugging Face is the central hub for the AI ecosystem. Our mission is to democratize good machine learning. We provide the tools and spaces for developers, researchers, and companies to accelerate their AI journey.

**Our Platform at a Glance:**
*   **2M+ Models:** Discover and use pre-trained models for every modality, including text, image, video, audio, and 3D.
*   **500k+ Datasets:** Access a vast collection of datasets to train and evaluate your models.
*   **1M+ AI Applications:** Explore, run, and share live AI demos in Spaces.

---

### For the Community: Create, Discover, Collaborate

For developers, researchers, and hobbyists, Hugging Face is your platform to innovate and share.

*   **Collaborate Freely:** Host and collaborate on unlimited public models, datasets, and applications.
*   **Move Faster:** Leverage the Hugging Face open-source stack to streamline your workflow.
*   **Build Your Portfolio:** Share your work with the world, gain visibility, and establish your profile in the ML community.

---

### For Business: Accelerate and Scale Your AI

We provide paid plans and enterprise-grade solutions to help your organization build AI with security, support, and control.

*   **Team & Enterprise Plans:** From small teams to large corporations, we offer plans that scale with your needs.
*   **Enterprise-Grade Security:** Integrate with your identity provider using Single Sign-On (SSO) for secure access.
*   **Compliance and Control:** Manage the location of your data with Regions and monitor activity with comprehensive Audit Logs.
*   **Dedicated Support:** Get expert help to accelerate your ML development and deployment.

---

### Join the Future of AI

Hugging Face is more than a platform; it's a movement built on collaboration, openness, and a passion for building the future of AI, together.

*   **Our Culture:** We believe in the power of community and open-source. Our culture is one of collaboration, innovation, and a shared drive to make AI accessible to everyone.
*   **Careers at Hugging Face:** For prospective recruits, joining our team means becoming part of a mission-driven company at the epicenter of the AI revolution. You will help build the tools that empower the entire industry.
*   **Get Involved:** Whether you are looking for your next career move, an investor seeking the leader in community-driven AI, or a customer ready to scale your ML efforts, we invite you to join us.

**Sign up and start building today!**

In [37]:
def stream_brochur(company_name,url):
    stream = gemini.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)    

In [38]:
stream_brochur("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Hugging Face is the central platform where the machine learning community collaborates on the cutting edge of artificial intelligence. We provide the tools and infrastructure for developers, researchers, and businesses to create, discover, and deploy AI models, datasets, and applications.

## Our Platform

Hugging Face offers a comprehensive ecosystem for all your AI needs:

*   **Models:** Explore over 2.8 million pre-trained models covering a vast range of tasks including text generation, image-to-text, text-to-speech, and more. Whether you're looking for models with billions of parameters or specialized libraries like Transformers and Diffusers, you'll find it here.
*   **Datasets:** Access nearly a million diverse datasets, spanning modalities from text and image to audio, video, and 3D. Filter by size, format, and type to find the perfect data for your project.
*   **Spaces:** Discover and run over a million AI applications. From AI interns that provide quick technical help to video generation and high-quality voice cloning, Spaces showcases the practical applications of AI.

## Accelerate Your ML Journey

With the Hugging Face open-source stack, you can move faster and build your machine learning projects more efficiently. We support a wide array of modalities, including text, image, video, audio, and even 3D.

## Join the Community

Hugging Face is more than just a platform; it's a vibrant community. Build your portfolio by sharing your work with the world and establishing your ML profile.

---

**For Prospective Customers:**
Leverage the vast resources of Hugging Face to accelerate your AI initiatives. Access state-of-the-art models and datasets, and deploy AI applications quickly and efficiently.

**For Investors:**
Hugging Face is at the forefront of the AI revolution, providing the foundational infrastructure for the global machine learning community. Our platform fosters innovation and collaboration, driving the future of AI.

**For Recruits:**
Join a dynamic and passionate team dedicated to building the future of AI. Be part of an open and collaborative environment where innovation thrives. Hugging Face offers exciting career opportunities for individuals passionate about machine learning and its potential.